## Setup Colab and sync


In [1]:
#@title Clone the repo and add API key
!git clone https://github.com/amusali/bert_for_patents.git
import os


Cloning into 'bert_for_patents'...
remote: Enumerating objects: 2333, done.
remote: Counting objects: 100% (591/591), done.
remote: Compressing objects: 100% (280/280), done.
remote: Total 2333 (delta 355), reused 514 (delta 290), pack-reused 1742 (from 2)
Receiving objects: 100% (2333/2333), 210.69 MiB | 40.75 MiB/s, done.
Resolving deltas: 100% (1314/1314), done.


In [2]:
#@title Pull latest changes from repo
%cd "/content/bert_for_patents"
!git pull origin master
#@title Add paths to the system and change directory
import sys
import os
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

/content/bert_for_patents
From https://github.com/amusali/bert_for_patents
 * branch            master     -> FETCH_HEAD
Already up to date.


In [3]:
#@title Run setup file to get environment
!python "/content/bert_for_patents/05 Analysis/01 Main/__ aux/setup_colab.py"

absl-py==2.0.0 is already installed.
Installing asttokens==2.4.1...
astunparse==1.6.3 is already installed.
attrs==26.1.0 is already installed.
beautifulsoup4==4.13.5 is already installed.
Installing bert==2.2.0...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for bert: filename=bert-2.2.0-py3-none-any.whl size=3816 sha256=c0d20b18ddb3d54447a988226318637bf63bbc0b147bace904f51b514842d988
  Stored in directory: /root/.cache/pip/wheels/cd/64/ff/6d3c6dba947a506f45d8ff1e5b60c219c3eba2949cb77fa4dc
  Created wheel for erlastic: filename=erlastic-2.0.0-py3-none-any.whl size=6839 sha256=82bb9c25a0a3df925ff13be4d27703dda72c64c2a2ddeb98229d1a5bd09fed26
  Stored in directory: /root/.cache/pip/wheels/10/bd/c6/7ad6665fdaa97581ae44881e02429c28e9e4cc80353eeed277
Suc

In [4]:
!ls

'00 Make patents data rectangular.ipynb'  '04 Colab'	 __init__.py
'00 Python data'			   api		 logdir
'01 Stata'				  '__ aux'	 path_utils.py
'03 Python scripts'			   config.json


In [5]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


Set precision

## Load the patents and process

In [6]:

import os
import sys
import json
import time
import numpy as np
import pandas as pd
import dill as pickle
import tensorflow as tf

from pathlib import Path
from torch.utils.data import Dataset, DataLoader, Subset

# Check GPU
print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("No GPU detected by TensorFlow!")

import api.bertembeddings as be

EMBEDDING_DIM = 1024


class PatentDataset(Dataset):
    def __init__(self, df):
        self.abstracts = df["abstract"].tolist()

    def __len__(self):
        return len(self.abstracts)

    def __getitem__(self, idx):
        return idx, self.abstracts[idx]


def collate_fn(batch):
    rows, abstracts = zip(*batch)

    embeddings = be.get_embd_of_whole_abstract(
        list(abstracts),
        return_cls_embedding=True,
        has_context_token=True
    )

    embeddings = np.asarray(
        embeddings, dtype=np.float16
    ).reshape(-1, EMBEDDING_DIM)

    return np.asarray(rows, dtype=np.int64), embeddings


def load(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)


TensorFlow: 2.21.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
BERT loaded...


In [7]:

# Load applications
path = "/content/drive/MyDrive/uc3m PhD/07 Continuous innovation space/02 Analysis/01 Main/01 Data load/out/02 All applications with an abstract - app_id as index.parquet"
df = pd.read_parquet(path, columns=['pgpub_id', 'patent_id', 'consolidated_abstract'])
df.rename(columns = {'consolidated_abstract':'abstract'}, inplace=True)
df["abstract_source"] = np.where(
    df["pgpub_id"].notna(),
    "pre-grant publication",
    "grant patent"
)


In [8]:

# Output files
SAVE_FOLDER = Path(
    "/content/drive/MyDrive/uc3m PhD/PhD Data/01 CLS Embeddings/Applications"
)
SAVE_FOLDER.mkdir(parents=True, exist_ok=True)

EMBEDDING_FILE = SAVE_FOLDER / "embeddings.npy"
INDEX_FILE = SAVE_FOLDER / "embedding_index.parquet"
PROGRESS_FILE = SAVE_FOLDER / "progress.json"
MODEL_INFO_FILE = SAVE_FOLDER / "model_info.json"

# Parameters
BATCH_SIZE = 256
SAVE_INTERVAL = 90 * 60

# Validate data
assert df.index.is_unique
assert df.index.notna().all()
assert df["abstract"].notna().all()
# Standardize ID types
df.index = df.index.astype("string")
df.index.name = "application_id"
df["patent_id"] = df["patent_id"].astype("string")

# Create the row-to-application mapping
index_df = df[["patent_id", "abstract_source"]].copy()

index_df.insert(0, "application_id", df.index.to_numpy())
index_df = index_df.reset_index(drop=True)

index_df.insert(
    0, "embedding_row", np.arange(len(df), dtype=np.int64)
)

# Model metadata
model_info = {
    "model_dir": str(be.MODEL_DIR),
    "vocab": str(be.VOCAB),
    "max_seq_length": be.MAX_SEQ_LENGTH,
    "embedding_dim": EMBEDDING_DIM,
    "context_token": "abstract",
    "storage_dtype": "float16",
    "tensorflow_version": tf.__version__,
}

# Check for existing run
files_exist = [
    EMBEDDING_FILE.exists(),
    INDEX_FILE.exists(),
    PROGRESS_FILE.exists(),
    MODEL_INFO_FILE.exists()
]


In [9]:

if all(files_exist):
    # Resume previous run
    old_index = pd.read_parquet(INDEX_FILE)
    old_model_info = json.loads(MODEL_INFO_FILE.read_text())

    if not old_index.equals(index_df):
        raise ValueError("Input data changed since last checkpoint!")

    if old_model_info != model_info:
        raise ValueError("Embedding model configuration changed!")

    processed_count = json.loads(
        PROGRESS_FILE.read_text()
    )["next_row"]

    embeddings = np.lib.format.open_memmap(
        EMBEDDING_FILE, mode="r+"
    )

    if embeddings.shape != (len(df), EMBEDDING_DIM):
        raise ValueError("Embedding array shape mismatch!")

    print(f"Resuming from row {processed_count:,}")

elif not any(files_exist):
    # New run
    index_df.to_parquet(INDEX_FILE, index=False)
    MODEL_INFO_FILE.write_text(json.dumps(model_info, indent=2))

    embeddings = np.lib.format.open_memmap(
        EMBEDDING_FILE,
        mode="w+",
        dtype=np.float16,
        shape=(len(df), EMBEDDING_DIM)
    )

    processed_count = 0
    embeddings.flush()
    PROGRESS_FILE.write_text(json.dumps({"next_row": 0}))

    print(f"New embedding array: {embeddings.shape}")

else:
    raise RuntimeError("Incomplete output files detected!")

# Save checkpoint
def save_checkpoint(next_row):
    embeddings.flush()

    temp_file = PROGRESS_FILE.with_suffix(".tmp")
    temp_file.write_text(json.dumps({"next_row": next_row}))
    os.replace(temp_file, PROGRESS_FILE)

    print(f"Checkpoint saved: {next_row:,}/{len(df):,}")

# DataLoader
dataset = PatentDataset(df)

dataloader = DataLoader(
    Subset(dataset, range(processed_count, len(dataset))),
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
    shuffle=False,
    num_workers=0
)

# Progress tracking
start_time = time.time()
last_save_time = start_time
last_progress_time = start_time
processed_in_last_minute = 0

print(f"Total applications: {len(df):,}")
print(f"Already processed: {processed_count:,}")
print(f"Remaining: {len(df) - processed_count:,}")


New embedding array: (11469409, 1024)
Total applications: 11,469,409
Already processed: 0
Remaining: 11,469,409


# TEST & COMPARISON RUN

In [8]:
# TEST RUN
import dill as pickle
# Load old embedding dictionary
old_path = "/content/drive/MyDrive/uc3m PhD/PhD Data/01 CLS Embeddings/All embeddings - float16/all_embeddings_float16.pkl"

# Load old embeddings
with open(old_path, "rb") as f:
    old_embeddings = pickle.load(f)


In [9]:

# Select 1000 patents available in both datasets
test_df = df[
    df["patent_id"].astype(str).isin(old_embeddings)
].sample(n=2000, random_state=42)

# Get old embeddings
old_array = np.stack([
    np.asarray(old_embeddings[pid]).reshape(1024)
    for pid in test_df["patent_id"]
])


In [12]:
import gc 
import itertools
old_embeddings = dict(itertools.islice(old_embeddings.items(), 10000))

In [14]:
len(old_embeddings)

10000

In [13]:
gc.collect()

15434

In [ ]:
with open("/content/drive/MyDrive/uc3m PhD/PhD Data/01 CLS Embeddings/All embeddings - float16/test_embeddings_float16.pkl", "wb") as f:
    pickle.dump(old_embeddings, f)


In [16]:
# Calculate new embeddings
new_batches = []

batch_size = 128

for i in range(0, len(test_df), batch_size):
    abstracts = test_df["abstract"].iloc[i:i + batch_size].tolist()

    new_batches.append(
        be.get_embd_of_whole_abstract(
            abstracts,
            return_cls_embedding=True,
            has_context_token=True
        )
    )

new_array = np.vstack(new_batches)

assert len(new_array) == len(test_df)

# Compare cosine similarity
from sklearn.metrics.pairwise import paired_cosine_distances

test_df["cosine_similarity"] = 1 - paired_cosine_distances(
    old_array.astype(np.float16),
    new_array.astype(np.float16)
)

print(test_df["cosine_similarity"].describe())

There are 128 patents being fed into BERT
It took 0.06149 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06153 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06140 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06156 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06151 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06154 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06165 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06162 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06153 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06161 seconds to get the embed

In [23]:
display(test_df.cosine_similarity.describe(percentiles=[0.01, 0.05]))

print(len(test_df[test_df.cosine_similarity < 0.9]) / len(test_df))

,cosine_similarity
count,2000.000000
mean,0.995163
std,0.025801
min,0.617446
1%,0.859265
5%,0.977116
50%,0.999999
max,1.000000


0.0155


In [18]:
test_df[["patent_id", "cosine_similarity"]].sort_values(
    "cosine_similarity"
).head(200)

,patent_id,cosine_similarity
application_id,,
11497926,7480593,0.617446
15380921,11447861,0.621531
16558597,11660590,0.700497
15550293,10218575,0.732825
14907396,9656995,0.769087
...,...,...
12612395,8260951,0.999998
09838981,7595914,0.999998
12270653,8332341,0.999998


# MAIN LOOP

In [10]:
#@title Main loop

try:
    for batch_rows, batch_embeddings in dataloader:

        batch_size = len(batch_rows)
        end = processed_count + batch_size

        # Verify correct row ordering
        expected = np.arange(processed_count, end)
        if not np.array_equal(batch_rows, expected):
            raise ValueError("Unexpected row ordering!")

        # Save embeddings directly into NumPy array
        embeddings[batch_rows] = batch_embeddings

        processed_count = end
        processed_in_last_minute += batch_size

        current_time = time.time()

        # Progress every minute
        if current_time - last_progress_time >= 30:

            elapsed = current_time - last_progress_time

            speed = processed_in_last_minute / elapsed

            print(
                f"Last minute: {processed_in_last_minute:,} | "
                f"Total: {processed_count:,}/{len(df):,} | "
                f"Speed: {speed:.1f} abstracts/sec"
            )

            last_progress_time = current_time
            processed_in_last_minute = 0

        # Checkpoint every 90 minutes
        if current_time - last_save_time >= SAVE_INTERVAL:
            save_checkpoint(processed_count)
            last_save_time = current_time

finally:
    # Save even if interrupted or an error occurs
    save_checkpoint(processed_count)

print("Embedding generation completed!")


There are 256 patents being fed into BERT
It took 0.03029 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.01456 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.01282 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.01289 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.01299 seconds to get the embeddings of the input.
Last minute: 1,280 | Total: 1,280/11,469,409 | Speed: 18.0 abstracts/sec
There are 256 patents being fed into BERT
It took 0.01304 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.01300 seconds to get the embeddings of the input.
Last minute: 512 | Total: 1,792/11,469,409 | Speed: 9.9 abstracts/sec
There are 256 patents being fed into BERT
It took 0.01318 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT

KeyboardInterrupt: 

In [ ]:
!pip install dill

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 10.8 MB/s eta 0:00:00


In [ ]:
print(len(embedding_dict))

6997638


# Combine with previously processed patents

In [ ]:
#@title Load all patents
previous_patents = load(r"/content/drive/MyDrive/PhD Data/01 CLS Embeddings/CheckedPatents_CLSonly_2024.11.11_00:06.pkl")
current_patents = load(r"/content/drive/MyDrive/PhD Data/01 CLS Embeddings/01 CLS embeddings rest - 20250226_163344.pkl")

assert(len(previous_patents) == 2077783)
assert(len(current_patents) == 6997638)


In [ ]:
#@title Combine, check and save

# Squeeze
for key, value in previous_patents.items():
    previous_patents[key] = value.squeeze()


# Combine
current_patents.update(previous_patents)
previous_patents = None



In [ ]:
# Get a list of all variables
all_vars = list(globals().keys())
print(all_vars)


['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', '_', '__', '___', '_i', '_ii', '_iii', '_i1', 'drive', '_i2', '_i3', 'time', 'pd', 'Dataset', 'DataLoader', 'os', '_exit_code', 'pickle', 'datetime', 'np', 'PatentDataset', 'collate_fn', 'save_embedding_dict', 'load', '_i4', 'previous_patents', 'current_patents', '_i5', 'key', 'value', '_i6', '_6', '_i7', '_7', '_i8', 'gc', 'sys', 'get_memory_variables', 'name', 'var', '_i9']


In [ ]:


#@title Check

#assert(len(current_patents) == 2777783 + 6997638)
print(len(current_patents))

# Check types
size_types  = set()
for key, value in current_patents.items():
    size_types.add(value.shape)

print(size_types)
assert(len(size_types) == 1)

9075421
{(1024,)}


In [ ]:
import dill as pickle
import os

# Define batch size
num_batches = 16  # Adjust based on testing
batch_size = len(current_patents) // num_batches

# Define save path (Change to your Google Drive path if using Colab)
save_dir = "/content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings"
os.makedirs(save_dir, exist_ok=True)

# Save in batches
for i in range(num_batches):
    start = i * batch_size
    end = (i + 1) * batch_size if i != num_batches - 1 else len(current_patents)

    batch_dict = dict(list(current_patents.items())[start:end])

    with open(f"{save_dir}/batch_{i}.pkl", "wb") as f:
        pickle.dump(batch_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Saved batch {i+1}/{num_batches} - Size: {len(batch_dict)} items")


Saved batch 1/16 - Size: 567213 items
Saved batch 2/16 - Size: 567213 items
Saved batch 3/16 - Size: 567213 items
Saved batch 4/16 - Size: 567213 items
Saved batch 5/16 - Size: 567213 items
Saved batch 6/16 - Size: 567213 items
Saved batch 7/16 - Size: 567213 items
Saved batch 8/16 - Size: 567213 items
Saved batch 9/16 - Size: 567213 items
Saved batch 10/16 - Size: 567213 items
Saved batch 11/16 - Size: 567213 items
Saved batch 12/16 - Size: 567213 items
Saved batch 13/16 - Size: 567213 items
Saved batch 14/16 - Size: 567213 items
Saved batch 15/16 - Size: 567213 items
Saved batch 16/16 - Size: 567226 items


# Combine, adjust precision and save to save memory

In [7]:
import os
import glob
import pickle
import numpy as np

# Define the folder containing your batched embeddings.
embedding_batches_folder = '/content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings'
# Define the output file path for the combined embeddings.
output_file = '/content/drive/MyDrive/PhD Data/01 CLS Embeddings/All embeddings - float16/all_embeddings_float16.pkl'

combined_embeddings = {}

# Get list of all batch files (assumed to have a .pkl extension)
batch_files = glob.glob(os.path.join(embedding_batches_folder, '*.pkl'))

print(f"Found {len(batch_files)} batch files. Processing...")

for batch_file in batch_files:
    print(f"Loading {batch_file}...")
    with open(batch_file, 'rb') as f:
        batch_dict = pickle.load(f)
    # Ensure keys are strings and values are NumPy arrays.
    for k, v in batch_dict.items():
        combined_embeddings[str(k)] = np.array(v)

print("All batch files loaded. Converting embeddings to float16...")

# Convert every embedding to float16.
for k in combined_embeddings:
    combined_embeddings[k] = combined_embeddings[k].astype(np.float16)

print("Conversion complete. Saving the combined embeddings to file...")

# Save the combined dictionary as a single pickle file.
with open(output_file, 'wb') as f:
    pickle.dump(combined_embeddings, f)

print(f"Combined embeddings saved to {output_file}.")

# Calculate the total size of combined_embeddings in bytes and print it in MB.
total_size_bytes = sum(emb.nbytes for emb in combined_embeddings.values())
print(f"Total size of combined_embeddings: {total_size_bytes / (1024**2):.2f} MB")



Found 16 batch files. Processing...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_0.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_1.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_2.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_3.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_4.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_5.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_6.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_7.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_8.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_9.pkl...
Loading /content/drive/MyDrive/PhD Data/01 C